In [0]:
%sql
CREATE OR REPLACE TABLE data_warehouse_factory.gold.dim_employee_assignments AS
WITH deduplicated_assignments AS (
  SELECT 
    assignment_key,
    upper(trim(line_code)) AS line_code,
    trim(line_name) AS line_name,
    upper(trim(cell_code)) AS cell_code,
    trim(cell_name) AS cell_name,
    
    default_employee_key,
    trim(default_employee_name) AS default_employee_name,
    backup_employee_key,
    trim(backup_employee_name) AS backup_employee_name,
    
    valid_from,
    upper(trim(shift)) AS shift,
    remarks,
    source,
    _silver_ingested_at,
    
    -- Wyznaczenie daty obowiązywania do (valid_to) na podstawie kolejnej zmiany w tej samej komórce
    LEAD(valid_from) OVER (
      PARTITION BY line_code, cell_code, shift 
      ORDER BY valid_from ASC
    ) AS next_valid_from
  FROM data_warehouse_factory.silver.silver_employee_assignments
)

SELECT 
  assignment_key,
  line_code,
  line_name,
  cell_code,
  cell_name,
  
  default_employee_key,
  default_employee_name,
  backup_employee_key,
  backup_employee_name,
  
  shift,
  valid_from,
  
  -- Jeśli brak nowszego wpisu, przypisanie obowiązuje do '9999-12-31'
  COALESCE(DATE_ADD(next_valid_from, -1), DATE '9999-12-31') AS valid_to,
  
  -- Flaga określająca, czy wpis jest aktualny
  CASE 
    WHEN next_valid_from IS NULL THEN TRUE 
    ELSE FALSE 
  END AS is_current,
  
  remarks,
  source,
  CURRENT_TIMESTAMP() AS _gold_created_at
FROM deduplicated_assignments;